In [ ]:
import wandb
import torch
import random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from pytorch_tabular.models.ft_transformer import FTTransformer
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

In [ ]:
wandb.init(project="FT-Transformer binData Intrusion Detection")

In [ ]:
data = pd.read_csv("../../dataStuff/UNSW_binData.csv")

In [ ]:
label_encoder = LabelEncoder()
data["label"] = label_encoder.fit_transform(data["label"])

In [ ]:
X = data.drop(columns=["label"]).values
y = data["label"].values.astype(float)

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train, X_val, y_train, y_val = (
    torch.tensor(X_train, dtype=torch.float32).to(device),
    torch.tensor(X_val, dtype=torch.float32).to(device),
    torch.tensor(y_train, dtype=torch.float32).to(device),
    torch.tensor(y_val, dtype=torch.float32).to(device),
)

In [ ]:
batch_size = 16
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

In [ ]:
model = FTTransformer(
    input_dim=X.shape[1],  # Number of features
    output_dim=1,  # Binary classification
    n_blocks=2,
    n_heads=4,
    attn_dropout=0.1,
    ff_dropout=0.1,
)
model.to(device)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-4)

In [ ]:
accuracy = BinaryAccuracy().to(device)
precision = BinaryPrecision().to(device)
recall = BinaryRecall().to(device)

In [ ]:
def train(model, train_loader):
    model.train()
    total_loss, total_acc = 0, 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch).squeeze(1)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy(outputs.sigmoid(), y_batch.int()).item()
    return total_loss / len(train_loader), total_acc / len(train_loader)


In [ ]:
def validate(model, val_loader):
    model.eval()
    total_loss, total_acc, total_prec, total_rec = 0, 0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            outputs = model(X_batch).squeeze(1)
            loss = criterion(outputs, y_batch)

            total_loss += loss.item()
            total_acc += accuracy(outputs.sigmoid(), y_batch.int()).item()
            total_prec += precision(outputs.sigmoid(), y_batch.int()).item()
            total_rec += recall(outputs.sigmoid(), y_batch.int()).item()
    
    return (total_loss / len(val_loader), total_acc / len(val_loader), 
            total_prec / len(val_loader), total_rec / len(val_loader))


In [ ]:
epochs = 10
for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader)
    val_loss, val_acc, val_prec, val_rec = validate(model, val_loader)

    dataDict = {
        "Epoch": epoch + 1,
        "Train Loss": train_loss,
        "Train Accuracy": train_acc,
        "Val Loss": val_loss,
        "Val Accuracy": val_acc,
        "Val Precision": val_prec,
        "Val Recall": val_rec,
    }
    
    print(dataDict)
    wandb.log(dataDict)

In [ ]:
torch.save(model.state_dict(), "ft_transformer_model.pth")
torch.save(scaler, "scaler.pth")
wandb.finish()